# ANN Operating Profit Prediction

This notebook predicts **Operating profit before tax** from annual enterprise survey data. The evaluation uses a **chronological split** so that later years are not used to train the model. Industry and enterprise-size categories are encoded without using the target column as an input.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
import joblib

tf.random.set_seed(42)
np.random.seed(42)


## 1. Load and clean the data


In [ ]:
DATA_PATH = "../data/annual-enterprise-survey-2025-financial-year-provisional-size-bands.csv"
df = pd.read_csv(DATA_PATH)

# Remove empty Excel-style columns.
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed:")], errors="ignore")

# Keep only the target measure.
profit_df = df[df["variable"].eq("Operating profit before tax")].copy()

# 'C' means confidential/suppressed and cannot be used as a numeric target.
profit_df["value"] = pd.to_numeric(profit_df["value"], errors="coerce")
profit_df = profit_df.dropna(subset=["value"])

# Exclude published aggregate rows. The model is intended to learn from size-band observations.
profit_df = profit_df[~profit_df["rme_size_grp"].isin(["i_Industry_Total", "j_Grand_Total"])].copy()

profit_df = profit_df.drop_duplicates()
print("Rows after cleaning:", len(profit_df))
print("Years:", profit_df["year"].min(), "to", profit_df["year"].max())
display(profit_df.head())


## 2. Define a time-based train/validation/test split

Using random rows would place observations from the same years/groups in both training and testing. Instead, the model trains on earlier years, validates on 2024, and tests on 2025. This gives a more realistic future-year evaluation.


In [ ]:
train_df = profit_df[profit_df["year"] <= 2023].copy()
val_df = profit_df[profit_df["year"] == 2024].copy()
test_df = profit_df[profit_df["year"] == 2025].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


## 3. Features and target

The target is operating profit. The input features are year, industry, and enterprise-size group. `industry_code_ANZSIC` is omitted because it duplicates the industry category information and would add unnecessary redundancy.


In [ ]:
FEATURES = ["year", "industry_name_ANZSIC", "rme_size_grp"]
TARGET = "value"

X_train = train_df[FEATURES]
y_train = train_df[TARGET].astype(float).to_numpy().reshape(-1, 1)
X_val = val_df[FEATURES]
y_val = val_df[TARGET].astype(float).to_numpy().reshape(-1, 1)
X_test = test_df[FEATURES]
y_test = test_df[TARGET].astype(float).to_numpy().reshape(-1, 1)

categorical_features = ["industry_name_ANZSIC", "rme_size_grp"]
numeric_features = ["year"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])

X_train_p = preprocessor.fit_transform(X_train)
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)

# Scale the target using training data only. This improves ANN regression stability.
target_scaler = StandardScaler()
y_train_s = target_scaler.fit_transform(y_train)
y_val_s = target_scaler.transform(y_val)
y_test_s = target_scaler.transform(y_test)

print("Encoded feature count:", X_train_p.shape[1])


## 4. Build and train the ANN


In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(X_train_p.shape[1],)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1)
])

model.compile(optimizer="adam", loss="mse", metrics=[keras.metrics.MeanAbsoluteError(name="mae")])

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=30, restore_best_weights=True
)

history = model.fit(
    X_train_p, y_train_s,
    validation_data=(X_val_p, y_val_s),
    epochs=300,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1,
)


## 5. Evaluate on the unseen 2025 test year


In [ ]:
pred_s = model.predict(X_test_p, verbose=0)
pred = target_scaler.inverse_transform(pred_s).ravel()
actual = y_test.ravel()

mae = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))
r2 = r2_score(actual, pred)

print(f"2025 Test MAE:  {mae:,.2f}")
print(f"2025 Test RMSE: {rmse:,.2f}")
print(f"2025 Test R²:   {r2:.4f}")


## 6. Training curve


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("ANN Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig("training_loss.png", dpi=300, bbox_inches="tight")
plt.show()


## 7. Actual vs predicted values


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(actual, pred, alpha=0.7)
lims = [min(actual.min(), pred.min()), max(actual.max(), pred.max())]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Actual Operating Profit")
plt.ylabel("Predicted Operating Profit")
plt.title("2025 Actual vs Predicted Operating Profit")
plt.tight_layout()
plt.savefig("actual_vs_predicted.png", dpi=300, bbox_inches="tight")
plt.show()


## 8. Save the model and preprocessing objects


In [ ]:
model.save("operating_profit_ann.keras")
joblib.dump(preprocessor, "preprocessor.pkl")
joblib.dump(target_scaler, "target_scaler.pkl")
profit_df.to_csv("cleaned_operating_profit_data.csv", index=False)
print("Saved model, preprocessors, cleaned data, and plots.")
